# Avance Fase 3 — Semana 2
## Núcleo algorítmico, eficiencia e implementación orientada a objetos

**Proyecto.** Factores asociados a la duración autorizada de los permisos de uso de vía
pública en San Francisco.

**Grupo 5** · Marco Álvarez Araya · María Belén Fassler Neumann · Ricardo Jaramillo Pulgar
**Curso** MCDI500 · Programación para la Ciencia de Datos · Magíster en Ciencia de Datos e
Inteligencia Artificial · Universidad Andrés Bello
**Docente** Dr. Omar Salinas Silva

---

### Qué hace esta fase

La Fase 2 dejó un *pipeline* que funciona y un conjunto procesado de 6.010 filas por 132
columnas. **Esta fase no construye un pipeline nuevo: reorganiza el mismo.** Lo que cambia es
la arquitectura del código, que pasa de celdas encadenadas a clases con estado.

El criterio para saber si la reorganización está bien hecha no es estético. Es verificable:
el pipeline de clases debe producir **exactamente** el mismo conjunto que la Fase 2. Esa
comprobación está en el apartado 8 y se ejecuta con `pd.testing.assert_frame_equal`.

### Recorrido

| § | Contenido |
|---|---|
| 1 | Contrato con las fases anteriores y carga del conjunto |
| 2 | Por qué pasar de funciones a objetos |
| 3 | La clase base `Transformador` y el contrato doble |
| 4 | Herencia: las siete clases hijas |
| 5 | Polimorfismo y la clase `Pipeline` |
| 6–7 | Etapa A y partición agrupada |
| 8 | **Verificación de equivalencia con la Fase 2** |
| 9 | La fuga de datos que la encapsulación reveló |
| 10 | La unidad de análisis: segmento o permiso |
| 11 | La transformación del objetivo, resuelta |
| 12 | Recursividad medida contra sus alternativas |
| 13 | Eficiencia: implementación propia contra scikit-learn, densa contra dispersa |
| 14 | Cohesión y acoplamiento |
| 15 | El patrón de diseño aplicado |
| 16 | Validación técnica |
| 17 | Documentación de la arquitectura y doble vía módulo/cuaderno |
| 18 | Persistencia y contrato con la Fase 4 |

### Qué se hereda de cada fase

- **De la Fase 1**: `src/utilidades.py`, `docs/diccionario_variables.csv` y
  `docs/metadatos_fase1.json`, que declaran la variable objetivo, los roles y los atributos
  prohibidos por fuga.
- **De la Fase 2**: `data/processed/permisos_procesado.csv` y `permisos_limpio.csv`, más las
  decisiones de imputación, codificación y escalamiento que aquí se encapsulan.
- **Del avance formativo**: las mediciones con `timeit`, `perf_counter` y `tracemalloc`, que
  aquí se aplican a decisiones reales del proyecto y no a ejemplos.

## Configuración

Una sola celda editable. Todo lo que depende del conjunto de datos vive aquí, de modo que
adaptar el cuaderno a otra fuente no obliga a recorrerlo entero. Los nombres replican los de
la Fase 2 para que la comparación entre ambos conjuntos sea directa.

In [1]:
# =====================================================================
# CONFIGURACIÓN DEL PROYECTO — editar solo esta celda
# =====================================================================
ARCHIVO_CRUDO = "Active_Street-Use_Permits_20260909.csv"
CORTE_DECLARADO = "2026-09-09"

OBJETIVO   = "duracion_dias"        # variable a explicar
OBJETIVO_LOG = "duracion_log"       # versión transformada, decidida en §11
GRUPO_VAR  = "permit_number"        # unidad de agrupación de la partición

# Variables por rol, tal como quedaron declaradas en la Fase 2
NOMINALES  = ["tipo_permiso", "agente", "distrito", "inspector",
              "analysis_neighborhood", "permit_zipcode"]
PREFIJOS   = {"tipo_permiso": "tipo", "agente": "agente", "distrito": "distrito",
              "inspector": "insp", "analysis_neighborhood": "barrio",
              "permit_zipcode": "zip"}
NUMERICAS  = ["n_segmentos", "lag_aprob_inicio", "anio_aprobacion", "mes_inicio",
              "largo_proposito"]
BINARIAS   = ["tiene_proposito"]

CATEGORICAS_IMPUTAR = ["permit_zipcode", "Cross Street 2", "analysis_neighborhood"]
NUMERICAS_IMPUTAR   = ["lag_aprob_inicio", "anio_aprobacion"]

PROHIBIDAS = ["Status", "permit_start_date", "permit_end_date"]   # fuga del objetivo

SEMILLA = 42                        # misma que en F1 y F2
PROPORCION_PRUEBA = 0.25            # misma que en F2

PROYECTO = {
    "nombre": "Factores asociados a la duración autorizada de permisos de vía pública",
    "grupo": "Grupo 5",
    "fase": "F3",
    "integrantes": ["Marco Álvarez Araya",
                    "María Belén Fassler Neumann",
                    "Ricardo Jaramillo Pulgar"],
    "repositorio": "https://github.com/RJaramilloP/proyecto-grupo5-mcdi500",
}
# =====================================================================

print(f"Objetivo declarado : {OBJETIVO}")
print(f"Agrupación         : {GRUPO_VAR}")
print(f"Semilla            : {SEMILLA}")
print(f"Nominales          : {len(NOMINALES)}  |  numéricas: {len(NUMERICAS)}  "
      f"|  binarias: {len(BINARIAS)}")

Objetivo declarado : duracion_dias
Agrupación         : permit_number
Semilla            : 42
Nominales          : 6  |  numéricas: 5  |  binarias: 1


## Preparación del entorno

Se declaran las librerías y se registra el intérprete efectivo. La verificación del entorno
es la misma de la Fase 1: si el cuaderno corre fuera del entorno virtual, lo dice en la
salida en lugar de fallar más adelante por una versión distinta.

`capturar()` reaparece aquí desde la Fase 2. Ejecuta una función, muestra su salida y además
la devuelve como texto, de modo que los anexos del informe se generan desde la propia
ejecución y no se transcriben a mano.

In [2]:
import contextlib
import io
import json
import math
import pickle
import platform
import sys
import time
import timeit
import tracemalloc
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

np.random.seed(SEMILLA)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)


def capturar(funcion, *args, **kwargs):
    """Ejecuta una función, muestra su salida y además la devuelve como texto."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        resultado = funcion(*args, **kwargs)
    texto = buffer.getvalue()
    print(texto, end="")
    return resultado, texto


def verificar_entorno():
    """Registra intérprete, plataforma y versiones instaladas."""
    en_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
    print(f"Intérprete      : {sys.executable}")
    print(f"Entorno virtual : {'[OK] activo' if en_venv else '[AVISO] parece el Python del sistema'}")
    print(f"Carpeta         : {Path.cwd()}")
    print(f"Sistema         : {platform.system()} {platform.release()}")
    print(f"Python          : {platform.python_version()}")
    print("\nLibrerías declaradas")
    import sklearn
    import scipy
    for mod in (np, pd, matplotlib, sklearn, scipy):
        print(f"  [OK]  {mod.__name__:<12} {mod.__version__}")


_, ANEXO_A3 = capturar(verificar_entorno)

Intérprete      : /usr/bin/python3
Entorno virtual : [AVISO] parece el Python del sistema
Carpeta         : /home/claude/limpio/F3
Sistema         : Linux 6.18.44-fc-v50
Python          : 3.11.15

Librerías declaradas
  [OK]  numpy        2.0.2
  [OK]  pandas       2.3.3
  [OK]  matplotlib   3.9.4
  [OK]  sklearn      1.6.1
  [OK]  scipy        1.13.1


## 1. Contrato con las fases anteriores

La Fase 2 estableció un contrato formal: importaba de la Fase 1 el diccionario y los
metadatos, y se detenía con una excepción si faltaba cualquiera de los dos. La coherencia
entre fases dejaba de ser una afirmación del informe y pasaba a ser una condición que el
código verifica.

**Esta fase extiende ese contrato a tres fuentes**, porque ahora hay más que heredar: el
diccionario y los metadatos de la Fase 1, los metadatos y el conjunto procesado de la Fase 2,
y el archivo original intacto. Si falta cualquiera, el proceso se detiene aquí y no produce
resultados parciales que después habría que rastrear.

In [3]:
MARCADORES = (".git", "requirements.txt", ".gitignore")


def localizar_raiz(inicio=None, marcadores=MARCADORES, niveles=5, respaldo=True):
    """Resuelve la raíz del proyecto subiendo hasta encontrar un marcador de repositorio.

    Evita rutas absolutas y `os.chdir()`: el cuaderno produce el mismo árbol de salidas
    sin importar desde qué carpeta se lance.
    """
    actual = Path(inicio or Path.cwd()).resolve()
    for candidata in [actual, *actual.parents][:niveles + 1]:
        if any((candidata / m).exists() for m in marcadores):
            return candidata, "estructurado"
    if respaldo:
        return actual, "plano"
    raise FileNotFoundError(f"No se encontró marcador de repositorio desde {actual}.")


RAIZ, MODO_RUTAS = localizar_raiz()
DIR_DATOS      = RAIZ / "data"
DIR_CRUDO      = DIR_DATOS / "raw"
DIR_PROCESADO  = DIR_DATOS / "processed"
DIR_DOCS       = RAIZ / "docs"
DIR_ANEXOS     = DIR_DOCS / "anexos"
DIR_SRC        = RAIZ / "src"
DIR_F3         = RAIZ / "F3"
DIR_ARTEFACTOS = RAIZ / "artefactos"

for carpeta in (DIR_PROCESADO, DIR_DOCS, DIR_ANEXOS, DIR_SRC, DIR_F3, DIR_ARTEFACTOS):
    carpeta.mkdir(parents=True, exist_ok=True)

if str(DIR_SRC) not in sys.path:
    sys.path.insert(0, str(DIR_SRC))
import utilidades

print(f"Raíz del proyecto : {RAIZ}")
print(f"Modo de rutas     : {MODO_RUTAS}")
print(f"Módulo compartido : utilidades desde {Path(utilidades.__file__).relative_to(RAIZ).as_posix()}")

Raíz del proyecto : /home/claude/limpio
Modo de rutas     : estructurado
Módulo compartido : utilidades desde src/utilidades.py


### 1.1 El contrato, verificado

Cinco artefactos obligatorios. La comprobación no avisa: **interrumpe**. Un cuaderno que
continúa con un contrato incompleto produce un resultado que parece válido y no lo es.

In [4]:
CONTRATO = {
    "diccionario F1":  DIR_DOCS / "diccionario_variables.csv",
    "metadatos F1":    DIR_DOCS / "metadatos_fase1.json",
    "metadatos F2":    DIR_DOCS / "metadatos_fase2.json",
    "procesado F2":    DIR_PROCESADO / "permisos_procesado.csv",
    "archivo original": DIR_CRUDO / ARCHIVO_CRUDO,
}

faltantes = [nombre for nombre, ruta in CONTRATO.items() if not ruta.exists()]
if faltantes:
    raise FileNotFoundError(
        "La Fase 3 no puede ejecutarse sin los artefactos de las fases previas. "
        f"Faltan: {faltantes}. Ejecute primero F1 y después F2.")

for nombre, ruta in CONTRATO.items():
    print(f"  [OK] {nombre:<18} {ruta.relative_to(RAIZ).as_posix():<52} "
          f"{ruta.stat().st_size:>9,} bytes")

diccionario = pd.read_csv(CONTRATO["diccionario F1"])
META_F1 = json.loads(CONTRATO["metadatos F1"].read_text(encoding="utf-8"))
META_F2 = json.loads(CONTRATO["metadatos F2"].read_text(encoding="utf-8"))

# Lo que se recupera del contrato, en vez de volver a declararlo a mano.
objetivo_declarado = diccionario.loc[diccionario.uso == "objetivo", "variable"].iloc[0]
prohibidas_declaradas = sorted(
    diccionario.loc[diccionario.uso.isin(["fuente_objetivo", "no_usada"]), "variable"])

print(f"\nObjetivo en el diccionario F1  : {objetivo_declarado}")
print(f"Objetivo en los metadatos F1   : {META_F1['objetivo']}")
print(f"Prohibidas por fuga (F1)       : {META_F1['prohibidas']}")
print(f"Componentes del objetivo (dic.): {prohibidas_declaradas}")
print(f"Variable de agrupación (F1)    : {META_F1['variable_grupo']}")
print(f"Filas del conjunto de trabajo  : F1 declara "
      f"{META_F1['conjunto_trabajo']['filas']:,}, F2 declara {META_F2['datos']['filas']:,}")
print(f"Semilla                        : F1={META_F1['semilla']}  "
      f"F2={META_F2['proyecto']['semilla']}  F3={SEMILLA}")

# El contrato no se declara: se comprueba. Cualquier discrepancia detiene la fase.
assert objetivo_declarado == META_F1["objetivo"] == OBJETIVO, (
    f"El objetivo difiere entre el diccionario, los metadatos y la configuración.")
assert META_F1["variable_grupo"] == GRUPO_VAR
assert sorted(META_F1["prohibidas"]) == sorted(PROHIBIDAS)
assert META_F1["conjunto_trabajo"]["filas"] == META_F2["datos"]["filas"]
assert META_F1["semilla"] == META_F2["proyecto"]["semilla"] == SEMILLA
print("\n[OK] Objetivo, agrupación, prohibidas, filas y semilla coinciden en las tres fases")

  [OK] diccionario F1     docs/diccionario_variables.csv                           2,049 bytes
  [OK] metadatos F1       docs/metadatos_fase1.json                                3,149 bytes
  [OK] metadatos F2       docs/metadatos_fase2.json                                1,579 bytes
  [OK] procesado F2       data/processed/permisos_procesado.csv                2,316,550 bytes
  [OK] archivo original   data/raw/Active_Street-Use_Permits_20260909.csv      3,397,697 bytes

Objetivo en el diccionario F1  : duracion_dias
Objetivo en los metadatos F1   : duracion_dias
Prohibidas por fuga (F1)       : ['Status', 'permit_start_date', 'permit_end_date']
Componentes del objetivo (dic.): ['Status', 'permit_end_date', 'permit_start_date']
Variable de agrupación (F1)    : permit_number
Filas del conjunto de trabajo  : F1 declara 6,010, F2 declara 6,010
Semilla                        : F1=42  F2=42  F3=42

[OK] Objetivo, agrupación, prohibidas, filas y semilla coinciden en las tres fases


### 1.2 Carga del archivo original

Se lee **el archivo local**, no el enlace de descarga. La fuente se actualiza a diario: una
nueva descarga produciría un conjunto distinto y haría el resultado no reproducible. El CSV
guardado en `data/raw/` es el dato, y la fecha de corte se verifica contra el sello
`data_as_of` que viene dentro del propio archivo en lugar de asumirse.

La lectura es **sin inferencia de tipos**. Es el hallazgo de la Fase 2: las coordenadas usan
coma decimal y una parte de las fechas admite dos lecturas, de modo que `read_csv` con tipos
inferidos produce un resultado equivocado sin emitir ninguna advertencia útil.

In [5]:
crudo = utilidades.leer_crudo(CONTRATO["archivo original"])

corte_real = pd.to_datetime(crudo["data_as_of"], format=utilidades.FORMATO_SELLO,
                            errors="coerce").max()
print(f"Archivo            : {ARCHIVO_CRUDO}")
print(f"Dimensiones        : {crudo.shape[0]:,} filas x {crudo.shape[1]} columnas")
print(f"Corte declarado    : {CORTE_DECLARADO}")
print(f"Corte en data_as_of: {corte_real.date()}")

assert str(corte_real.date()) == CORTE_DECLARADO, (
    f"El archivo dice {corte_real.date()} y la configuración {CORTE_DECLARADO}.")
print("\n[OK] La fecha de corte se verificó contra el propio archivo, no se supuso.")

Archivo            : Active_Street-Use_Permits_20260909.csv
Dimensiones        : 6,852 filas x 29 columnas
Corte declarado    : 2026-09-09
Corte en data_as_of: 2026-09-09

[OK] La fecha de corte se verificó contra el propio archivo, no se supuso.


## 2. De funciones a objetos: por qué

La Fase 2 preparó los datos con funciones y celdas encadenadas. Funciona, y el resultado es
correcto. El problema aparece cuando hay que responder preguntas sobre ese resultado.

In [6]:
# El estilo de la Fase 2, reducido a lo esencial: cada paso depende del anterior
# y lo aprendido en cada uno queda en variables sueltas del cuaderno.
demo = utilidades.convertir_tipos(crudo.head(200))
mediana_lag = demo["lag_aprob_inicio"].median() if "lag_aprob_inicio" in demo else None

print("Estilo funcional de la Fase 2:")
print("  df = convertir_tipos(crudo)")
print("  df = derivar_variables(df)")
print("  df['permit_zipcode'] = df['permit_zipcode'].fillna('Desconocido')")
print("  ...")
print("\nPreguntas que este estilo no puede responder sin volver a ejecutar todo:")
for pregunta in ["¿con qué mediana se imputó lag_aprob_inicio?",
                 "¿qué categorías conocía el codificador al ajustarse?",
                 "¿ese valor se aprendió de entrenamiento o de todo el conjunto?",
                 "¿puedo aplicar exactamente la misma transformación a datos nuevos?"]:
    print(f"  - {pregunta}")

Estilo funcional de la Fase 2:
  df = convertir_tipos(crudo)
  df = derivar_variables(df)
  df['permit_zipcode'] = df['permit_zipcode'].fillna('Desconocido')
  ...

Preguntas que este estilo no puede responder sin volver a ejecutar todo:
  - ¿con qué mediana se imputó lag_aprob_inicio?
  - ¿qué categorías conocía el codificador al ajustarse?
  - ¿ese valor se aprendió de entrenamiento o de todo el conjunto?
  - ¿puedo aplicar exactamente la misma transformación a datos nuevos?


Las cuatro preguntas anteriores tienen algo en común: todas son sobre **el estado
aprendido**, y en el estilo funcional ese estado no existe como objeto. Vive disperso en
variables del cuaderno, o se recalcula en cada llamada.

| Problema del estilo funcional | Consecuencia concreta en este proyecto |
|---|---|
| El estado aprendido no se guarda | La mediana de imputación quedó en una variable suelta; para saberla hay que releer la celda |
| El orden es implícito | Nada impide escalar antes de imputar; el error aparecería en un gráfico, no en una excepción |
| No se distingue ajuste de aplicación | **Es exactamente así como se filtra información de prueba al entrenamiento** |
| Cada paso se prueba entero o no se prueba | Probar la codificación obliga a ejecutar la conversión de tipos y la derivación |

La tercera fila no es teórica. En el apartado 9 se muestra que la Fase 2 incurrió en ella, y
que la encapsulación fue lo que lo dejó a la vista.

## 3. La clase base `Transformador`

Todos los pasos del *pipeline* tienen la misma forma: miran una o varias columnas, aprenden
algo de ellas y modifican el conjunto. Ese ochenta por ciento común se escribe **una vez**.

Tres mecanismos del lenguaje sostienen el diseño:

- **`@property` sin *setter*.** Convierte un método en algo que se lee como atributo pero no
  se puede asignar desde fuera. `paso.parametros` devuelve una **copia**: quien la recibe no
  puede alterar el estado interno sin que la clase se entere.
- **El guion bajo inicial.** `_parametros` y `_ajustado` son internos por convención. Python
  no lo impide; la convención avisa a quien lee el código.
- **`NotImplementedError`.** Deja el contrato explícito: si una clase hija olvida implementar
  `aprender` o `aplicar`, el error lo dice con claridad en vez de fallar veinte líneas
  después.

### 3.1 El contrato doble, y por qué no es redundante

La clase expone **dos juegos de nombres para la misma maquinaria**: `ajustar`/`transformar`,
que es el contrato del material del curso, y `fit`/`transform`, que es la interfaz de
scikit-learn. Los segundos **delegan** en los primeros: no hay dos implementaciones que
puedan desincronizarse.

La razón es concreta. La Fase 2 dejó `ImputadorMixto` escrito con `fit`/`transform` y persistió
un transformador ajustado con esa interfaz. Romperla obligaría a reajustar objetos que ya
aprendieron sus parámetros. Mantener ambas permite componer sin reentrenar, y deja la
decisión sujeta a medición en vez de a preferencia: en el apartado 13 se compara la
implementación propia contra la de scikit-learn en tiempo, memoria y equivalencia exacta.

In [7]:
# >>> src/transformadores.py
# Constantes heredadas de la Fase 1. No se redefinen aquí: se toman del módulo
# compartido, de modo que exista UNA sola declaración de cada una en el proyecto.
FORMATO_FECHA   = utilidades.FORMATO_FECHA      # "%m/%d/%Y", declarado, nunca inferido
FORMATO_SELLO   = utilidades.FORMATO_SELLO
COLS_FECHA      = utilidades.COLS_FECHA
COLS_COORDENADA = utilidades.COLS_COORDENADA
DESCARTES       = utilidades.DESCARTES          # 14 columnas y su motivo

In [8]:
# >>> src/transformadores.py
class Transformador:
    """Clase base de todo paso del pipeline. No se usa directamente.

    Define QUE metodos tiene cada paso y CUANDO se ejecutan; las clases hijas
    definen QUE aprende cada uno y COMO lo aplica.

    Contrato doble
    --------------
    Se exponen dos juegos de nombres para la misma maquinaria:

    - ``ajustar`` / ``transformar``: el contrato del material del curso.
    - ``fit`` / ``transform``: la interfaz de scikit-learn, para que los
      objetos ya ajustados de la Fase 2 sigan componiendo sin reescribirse.

    No son dos implementaciones: los segundos delegan en los primeros. La
    equivalencia se mide en la seccion 11 del cuaderno.
    """

    def __init__(self, columnas=None):
        # Atributo publico: parte de la configuracion, cualquiera puede leerlo.
        self.columnas = ([columnas] if isinstance(columnas, str)
                         else list(columnas) if columnas is not None else [])
        # Guion bajo inicial: estado INTERNO. Python no lo impide, pero la
        # convencion avisa a quien lee que no debe tocarse desde fuera.
        self._parametros = {}
        self._ajustado = False

    # ---- lectura controlada del estado ---------------------------------
    @property
    def nombre(self):
        """Identifica al objeto por su clase real, no por la clase base."""
        detalle = ", ".join(self.columnas) if self.columnas else ""
        return f"{type(self).__name__}({detalle})" if detalle else type(self).__name__

    @property
    def parametros(self):
        """Copia defensiva: quien la recibe no puede alterar el estado interno."""
        return dict(self._parametros)

    @property
    def ajustado(self):
        return self._ajustado

    # ---- el ciclo de vida ----------------------------------------------
    def ajustar(self, datos):
        """Aprende los parametros del subconjunto recibido."""
        self.verificar_columnas(datos)
        self._parametros = self.aprender(datos)
        self._ajustado = True
        return self                      # permite encadenar

    def transformar(self, datos):
        """Aplica lo aprendido. Falla si todavia no se ajusto."""
        if not self._ajustado:
            raise RuntimeError(f"{self.nombre}: hay que ajustar antes de transformar.")
        return self.aplicar(datos.copy())   # copia: nunca se altera la entrada

    def ajustar_transformar(self, datos):
        return self.ajustar(datos).transformar(datos)

    # ---- alias con la interfaz de scikit-learn --------------------------
    def fit(self, X, y=None):
        return self.ajustar(X)

    def transform(self, X):
        return self.transformar(X)

    def fit_transform(self, X, y=None):
        return self.ajustar_transformar(X)

    # ---- puntos de extension que cada hija implementa -------------------
    def verificar_columnas(self, datos):
        """Valida la entrada antes de trabajar, para fallar donde se entiende."""
        faltan = [c for c in self.columnas if c not in datos.columns]
        if faltan:
            raise KeyError(f"{self.nombre}: columnas ausentes: {sorted(faltan)}")

    def aprender(self, datos):
        raise NotImplementedError("Cada clase hija debe implementar aprender().")

    def aplicar(self, datos):
        raise NotImplementedError("Cada clase hija debe implementar aplicar().")

    def __repr__(self):
        return f"<{self.nombre} {'ajustado' if self._ajustado else 'sin ajustar'}>"


# =====================================================================
# ETAPA A: antes de la particion (no hay fuga posible)
# =====================================================================

### 3.2 La clase base no se usa sola

Dos comprobaciones inmediatas: la base no sabe qué aprender, y ningún paso puede transformar
antes de ajustarse.

In [9]:
class _Ejemplo(Transformador):
    """Clase mínima para comprobar el contrato de la base."""
    def aprender(self, datos):
        return {"n": len(datos)}
    def aplicar(self, datos):
        datos["marca"] = self._parametros["n"]
        return datos


tabla_demo = pd.DataFrame({"a": [1, 2, 3]})

# 1. La base no implementa aprender(): el contrato lo dice explícitamente
try:
    Transformador(["a"]).ajustar(tabla_demo)
except NotImplementedError as error:
    print("NotImplementedError :", error)

# 2. Transformar antes de ajustar está prohibido por la guarda de estado
try:
    _Ejemplo(["a"]).transformar(tabla_demo)
except RuntimeError as error:
    print("RuntimeError        :", error)

# 3. El estado interno solo se lee, no se escribe desde fuera
paso = _Ejemplo(["a"]).ajustar(tabla_demo)
copia = paso.parametros
copia["n"] = 999                       # modificar la copia no toca el objeto
print(f"parámetros del objeto: {paso.parametros}   (la copia decía {copia})")
try:
    paso.parametros = {"n": 0}
except AttributeError as error:
    print("AttributeError      :", error)

# 4. Los dos contratos son el MISMO objeto, no dos implementaciones
a = _Ejemplo(["a"]).ajustar_transformar(tabla_demo)
b = _Ejemplo(["a"]).fit_transform(tabla_demo)
pd.testing.assert_frame_equal(a, b)
print("\n[OK] ajustar_transformar y fit_transform dan el mismo resultado")

NotImplementedError : Cada clase hija debe implementar aprender().
RuntimeError        : _Ejemplo(a): hay que ajustar antes de transformar.
parámetros del objeto: {'n': 3}   (la copia decía {'n': 999})
AttributeError      : property 'parametros' of '_Ejemplo' object has no setter

[OK] ajustar_transformar y fit_transform dan el mismo resultado


## 4. Herencia: las siete clases hijas

Cada hija implementa **solo** `aprender` y `aplicar`. Todo lo demás —la validación de
entrada, la copia defensiva, el control de estado, los dos contratos— lo recibe de la base
sin volver a escribirlo.

Las clases se agrupan en dos etapas separadas por la partición, y esa separación no es
cosmética: **la etapa A puede aprender de todo el conjunto porque la partición aún no
existe; la etapa B no.**

### 4.1 Etapa A — antes de la partición

`SaneadorConjunto` es el único paso que cambia el número de filas. Por eso va antes de
partir: después, alterar filas invalidaría las máscaras de entrenamiento y prueba.

In [10]:
# >>> src/transformadores.py
class ConversorTipos(Transformador):
    """Convierte coordenadas, fechas y centinelas con formato declarado.

    Encapsula el hallazgo de la Fase 2: las coordenadas usan coma decimal y el
    formato de fecha admite dos lecturas. `aprender` deja registrado cuanto se
    convirtio, de modo que el diagnostico queda en el estado del objeto.
    """

    def aprender(self, datos):
        registro = {}
        for col in COLS_COORDENADA:
            if col in datos.columns:
                directo = pd.to_numeric(datos[col], errors="coerce").notna().sum()
                con_punto = pd.to_numeric(
                    datos[col].str.replace(",", ".", regex=False), errors="coerce"
                ).notna().sum()
                registro[f"{col}_sin_correccion"] = int(directo)
                registro[f"{col}_con_correccion"] = int(con_punto)
        for col in COLS_FECHA:
            if col in datos.columns:
                explicito = pd.to_datetime(datos[col], format=FORMATO_FECHA,
                                           errors="coerce").notna().sum()
                registro[f"{col}_formato_explicito"] = int(explicito)
        return registro

    def aplicar(self, datos):
        for col in COLS_COORDENADA:
            if col in datos.columns:
                datos[col] = pd.to_numeric(
                    datos[col].str.replace(",", ".", regex=False), errors="coerce")
        for col in COLS_FECHA:
            if col in datos.columns:
                datos[col] = pd.to_datetime(datos[col], format=FORMATO_FECHA,
                                            errors="coerce")
        if "data_as_of" in datos.columns:
            datos["data_as_of"] = pd.to_datetime(datos["data_as_of"],
                                                 format=FORMATO_SELLO, errors="coerce")
        if "permit_zipcode" in datos.columns:
            datos["permit_zipcode"] = datos["permit_zipcode"].replace("0", np.nan)
        return datos

In [11]:
# >>> src/transformadores.py
class DerivadorVariables(Transformador):
    """Deriva el objetivo y las explicativas, y agrupa las nominales largas.

    El agrupamiento de `Permit Type` y `Agent` en sus niveles mas frecuentes es
    estado aprendido, no una constante: por eso vive en `_parametros` y no en el
    cuerpo de `aplicar`.
    """

    REQUERIDAS = ("permit_start_date", "permit_end_date", "Approved Date",
                  "Permit Type", "Agent", "supervisor_district", "permit_number")

    def __init__(self, n_tipos=12, n_agentes=12):
        super().__init__(columnas=list(self.REQUERIDAS))
        self.n_tipos = n_tipos
        self.n_agentes = n_agentes

    def aprender(self, datos):
        return {
            "tipos_principales": list(
                datos["Permit Type"].value_counts().head(self.n_tipos).index),
            "agentes_principales": list(
                datos["Agent"].value_counts().head(self.n_agentes).index),
        }

    def aplicar(self, datos):
        datos["duracion_dias"] = (datos["permit_end_date"]
                                  - datos["permit_start_date"]).dt.days
        datos["n_segmentos"] = datos.groupby("permit_number")["permit_number"].transform("size")
        datos["lag_aprob_inicio"] = (datos["permit_start_date"]
                                     - datos["Approved Date"]).dt.days
        datos["anio_aprobacion"] = datos["Approved Date"].dt.year
        datos["mes_inicio"] = datos["permit_start_date"].dt.month

        datos["tipo_permiso"] = np.where(
            datos["Permit Type"].isin(self._parametros["tipos_principales"]),
            datos["Permit Type"], "Otros")
        datos["agente"] = np.where(
            datos["Agent"].isin(self._parametros["agentes_principales"]),
            datos["Agent"].fillna("Desconocido"), "Otros")
        datos["distrito"] = datos["supervisor_district"].fillna("Desconocido").astype(str)
        return datos


class SaneadorConjunto(Transformador):
    """Descarta columnas sin valor analitico y filas sin variable objetivo.

    Es el unico paso que cambia el numero de filas, y por eso se ejecuta antes
    de la particion: despues, alterar filas invalidaria las mascaras.
    """

    def __init__(self, descartes=None, objetivo="duracion_dias"):
        super().__init__()
        self.descartes = dict(descartes if descartes is not None else DESCARTES)
        self.objetivo = objetivo

    def aprender(self, datos):
        presentes = [c for c in self.descartes if c in datos.columns]
        return {"descartadas": presentes, "filas_entrada": int(len(datos))}

    def aplicar(self, datos):
        datos = datos.drop(columns=self._parametros["descartadas"])
        datos = datos.dropna(subset=[self.objetivo])
        datos = datos.drop_duplicates()
        return datos.reset_index(drop=True)


# =====================================================================
# ETAPA B: despues de la particion (aqui si puede haber fuga)
# =====================================================================

### 4.2 Etapa B — después de la partición

Aquí cada parámetro aprendido es una oportunidad de fuga. `DerivadorAusencia` aprende qué
inspectores son frecuentes; `ImputadorMixto`, las medianas; `CodificadorNominal`, el
vocabulario de categorías; `EscaladorEstandar`, medias y desviaciones. **Si cualquiera de
esos cuatro aprende del conjunto completo, información de prueba entra al entrenamiento.**

`CodificadorNominal` merece atención: recorre el vocabulario **aprendido**, no las categorías
del conjunto que está transformando. Una categoría que solo aparece en prueba no genera
columna, porque el modelo nunca pudo aprender nada de ella.

In [12]:
# >>> src/transformadores.py
class DerivadorAusencia(Transformador):
    """Codifica la ausencia informativa como variable, en vez de imputarla.

    Materializa el hallazgo de la Fase 2: donde falta el proposito la duracion
    mediana es casi nueve veces mayor, de modo que el hueco es senal y no ruido.
    """

    def __init__(self, n_inspectores=12):
        super().__init__(columnas=["Permit Purpose", "Inspector"])
        self.n_inspectores = n_inspectores

    def aprender(self, datos):
        return {"inspectores_principales":
                list(datos["Inspector"].value_counts().head(self.n_inspectores).index)}

    def aplicar(self, datos):
        datos["tiene_proposito"] = datos["Permit Purpose"].notna().astype(int)
        datos["largo_proposito"] = datos["Permit Purpose"].fillna("").str.len()
        principales = self._parametros["inspectores_principales"]
        datos["inspector"] = np.where(
            datos["Inspector"].isna(), "Sin asignar",
            np.where(datos["Inspector"].isin(principales), datos["Inspector"], "Otros"))
        return datos


class ImputadorMixto(Transformador):
    """Imputa categoricas con etiqueta explicita y numericas con la mediana.

    Heredada de la Fase 2. Alli era una clase suelta con interfaz fit/transform;
    aqui hereda el contrato comun y deja de repetir el control de estado.
    """

    def __init__(self, categoricas, numericas, etiqueta="Desconocido"):
        super().__init__(columnas=list(categoricas) + list(numericas))
        self.categoricas = list(categoricas)
        self.numericas = list(numericas)
        self.etiqueta = etiqueta

    def aprender(self, datos):
        return {"medianas": {c: float(datos[c].median()) for c in self.numericas}}

    def aplicar(self, datos):
        for col in self.categoricas:
            datos[col] = datos[col].fillna(self.etiqueta)
        for col in self.numericas:
            datos[col] = datos[col].fillna(self._parametros["medianas"][col])
        return datos

In [13]:
# >>> src/transformadores.py
class CodificadorNominal(Transformador):
    """Convierte una variable nominal en columnas indicadoras 0/1.

    El vocabulario se APRENDE en el ajuste. Una categoria que solo aparece en
    prueba no genera columna, porque el modelo no pudo aprender nada de ella.
    """

    def __init__(self, columna, prefijo=None):
        super().__init__(columnas=[columna])
        self.columna = columna
        self.prefijo = prefijo or columna

    def aprender(self, datos):
        return {"categorias": sorted(datos[self.columna].astype(str).unique())}

    @property
    def columnas_creadas(self):
        return [f"{self.prefijo}_{utilidades.normalizar_nombre(str(c))}"
                for c in self._parametros.get("categorias", [])]

    def aplicar(self, datos):
        valores = datos[self.columna].astype(str)
        for categoria, etiqueta in zip(self._parametros["categorias"],
                                       self.columnas_creadas):
            datos[etiqueta] = (valores == categoria).astype(int)
        return datos


class EscaladorEstandar(Transformador):
    """Centra en cero y escala a desviacion uno, con media y desviacion aprendidas.

    Implementacion propia, equivalente a StandardScaler. Mantener las dos permite
    compararlas en tiempo y memoria (seccion 11) en vez de suponer cual conviene.
    """

    def aprender(self, datos):
        medias, desviaciones = {}, {}
        for col in self.columnas:
            serie = pd.to_numeric(datos[col], errors="coerce")
            medias[col] = float(serie.mean())
            # ddof=0: misma convencion que StandardScaler, que divide por n.
            desv = float(serie.std(ddof=0))
            desviaciones[col] = desv if desv != 0 else 1.0
        return {"medias": medias, "desviaciones": desviaciones}

    def aplicar(self, datos):
        for col in self.columnas:
            datos[col] = ((pd.to_numeric(datos[col], errors="coerce")
                           - self._parametros["medias"][col])
                          / self._parametros["desviaciones"][col])
        return datos

### 4.3 `AgregadorPermiso`: la clase propia de esta fase

Las seis clases anteriores encapsulan trabajo que la Fase 2 ya hacía. Esta es nueva, y
materializa la decisión de unidad de análisis que se justifica con datos en el apartado 10.

Lleva el conjunto del **segmento de calle** al **permiso**. Verifica primero que las columnas
declaradas constantes lo sean de verdad —si no lo son, lanza `ValueError` en vez de promediar
en silencio— y conserva la variación que se pierde al agregar en indicadores `cruza_*`
explícitos.

In [14]:
# >>> src/transformadores.py
class AgregadorPermiso(Transformador):
    """Lleva el conjunto del segmento de calle al permiso.

    Es la pieza que materializa la decision de unidad de analisis de la Fase 3.
    El objetivo y once de las dieciocho variables son constantes dentro del
    permiso, de modo que repetir la fila no aporta informacion sobre la duracion:
    solo reparte peso. Las cuatro variables que si varian entre segmentos se
    resumen por su valor mas frecuente, y la variacion perdida se conserva en
    indicadores `cruza_*` explicitos.
    """

    def __init__(self, grupo="permit_number", constantes=(), modales=(),
                 medias=(), conservar=()):
        super().__init__(columnas=[grupo])
        self.grupo = grupo
        self.constantes = list(constantes)
        self.modales = list(modales)
        self.medias = list(medias)
        self.conservar = list(conservar)

    def aprender(self, datos):
        no_constantes = [c for c in self.constantes
                         if c in datos.columns
                         and datos.groupby(self.grupo)[c].nunique(dropna=False).gt(1).any()]
        return {"grupos": int(datos[self.grupo].nunique()),
                "filas_entrada": int(len(datos)),
                "constantes_violadas": no_constantes}

    @staticmethod
    def _moda(serie):
        modas = serie.mode(dropna=False)
        return modas.iloc[0] if len(modas) else np.nan

    def aplicar(self, datos):
        if self._parametros["constantes_violadas"]:
            raise ValueError(
                f"{self.nombre}: estas columnas no son constantes dentro del grupo: "
                f"{self._parametros['constantes_violadas']}")
        agrupado = datos.groupby(self.grupo, sort=False)
        partes = {}
        for col in self.constantes + self.conservar:
            if col in datos.columns:
                partes[col] = agrupado[col].first()
        for col in self.modales:
            if col in datos.columns:
                partes[col] = agrupado[col].agg(self._moda)
                partes[f"cruza_{col}"] = agrupado[col].nunique(dropna=False).gt(1).astype(int)
        for col in self.medias:
            if col in datos.columns:
                partes[col] = agrupado[col].mean()
        resultado = pd.DataFrame(partes)
        return resultado.reset_index()


# =====================================================================
# PATRON STRATEGY: varias formas de rellenar un hueco
# =====================================================================

## 5. Polimorfismo y la clase `Pipeline`

**Polimorfismo** es que quien usa estas clases no necesita saber de qué tipo es cada una.
El `Pipeline` llama `paso.ajustar(...)` y `paso.transformar(...)` sin preguntar nada, y por
eso agregar un transformador nuevo no obliga a modificar el `Pipeline`.

El argumento `mascara` es el elemento central del diseño y conviene leerlo con cuidado:

- `ajustar(datos)` aprende de **todo** lo que recibe.
- `ajustar(datos, mascara)` aprende **solo** de las filas marcadas y aplica a todas.

Esa diferencia de un argumento es la que separa un *pipeline* con fuga de uno sin ella. En el
estilo funcional era una decisión repetida en cada llamada y fácil de olvidar; aquí es un
parámetro único, explícito y verificable.

`Pipeline` vive en `src/pipeline.py`, separado de `src/transformadores.py`. La separación es
deliberada: **este módulo sabe cómo encadenar pasos, pero no sabe qué hace ninguno.** Esa es
la prueba del bajo acoplamiento del apartado 14.

In [15]:
# >>> src/pipeline.py
class Pipeline:
    """Encadena transformadores y controla el orden de ajuste.

    La mascara de ajuste es el elemento central del diseno. `ajustar(datos)`
    aprende de todo lo que recibe; `ajustar(datos, mascara)` aprende solo de las
    filas marcadas y aplica a todas. Esa diferencia de un argumento es la que
    separa un pipeline con fuga de uno sin ella, y aqui es explicita y medible.
    """

    def __init__(self, pasos=None, nombre="pipeline"):
        self._pasos = []
        self._ajustado = False
        self.nombre = nombre
        for paso in (pasos or []):
            self.agregar(paso)

    def agregar(self, transformador):
        if not isinstance(transformador, Transformador):
            raise TypeError(f"{self.nombre}: se esperaba un Transformador y llego "
                            f"{type(transformador).__name__}.")
        self._pasos.append(transformador)
        return self

    def ajustar(self, datos, mascara=None):
        """Ajusta cada paso y propaga el resultado al siguiente.

        Parametros
        ----------
        datos : pd.DataFrame
        mascara : pd.Series of bool | None
            Filas con las que se aprende. None significa aprender de todo, lo
            que solo es legitimo antes de la particion.
        """
        intermedio = datos.copy()
        for paso in self._pasos:
            if mascara is None:
                base = intermedio
            else:
                if len(mascara) != len(intermedio):
                    raise ValueError(
                        f"{self.nombre}: la mascara tiene {len(mascara)} filas y el "
                        f"conjunto {len(intermedio)}. Un paso cambio el numero de filas "
                        f"despues de la particion.")
                base = intermedio.loc[mascara.to_numpy()]
            paso.ajustar(base)                      # polimorfismo: no se pregunta el tipo
            intermedio = paso.transformar(intermedio)
        self._ajustado = True
        return self

    def transformar(self, datos):
        if not self._ajustado:
            raise RuntimeError(f"{self.nombre}: hay que ajustar antes de transformar.")
        resultado = datos.copy()
        for paso in self._pasos:
            resultado = paso.transformar(resultado)
        return resultado

    def ajustar_transformar(self, datos, mascara=None):
        return self.ajustar(datos, mascara).transformar(datos)

    # ---- alias scikit-learn --------------------------------------------
    def fit(self, X, y=None):
        return self.ajustar(X)

    def transform(self, X):
        return self.transformar(X)

    def fit_transform(self, X, y=None):
        return self.ajustar_transformar(X)

    # ---- introspeccion --------------------------------------------------
    def pasos_ejecutados(self):
        return tuple(self._pasos)

    def resumen(self):
        return pd.DataFrame([
            {"orden": i, "paso": p.nombre, "clase": type(p).__name__,
             "ajustado": p.ajustado, "n_parametros": len(p.parametros)}
            for i, p in enumerate(self._pasos, start=1)])

    def __len__(self):
        return len(self._pasos)

    def __repr__(self):
        estado = "ajustado" if self._ajustado else "sin ajustar"
        return f"Pipeline('{self.nombre}', {len(self._pasos)} pasos, {estado})"

### 5.1 El bucle polimórfico, en evidencia

Tres clases distintas, la misma llamada. Ninguna línea del `Pipeline` pregunta de qué tipo es
el paso que está ejecutando.

In [16]:
pasos_demo = [ConversorTipos(), DerivadorVariables(), SaneadorConjunto()]

print("Tres clases distintas, la misma interfaz:\n")
for paso in pasos_demo:
    print(f"  {type(paso).__name__:<20} hereda de {type(paso).__bases__[0].__name__:<14} "
          f"implementa aprender={'aprender' in type(paso).__dict__} "
          f"aplicar={'aplicar' in type(paso).__dict__}")

print("\nY el Pipeline los recorre sin preguntar el tipo de ninguno:")
print("    for paso in self._pasos:")
print("        paso.ajustar(base)")
print("        intermedio = paso.transformar(intermedio)")

# Herencia efectiva: qué clases descienden del contrato
print(f"\nSubclases de Transformador registradas: {len(Transformador.__subclasses__())}")
for clase in Transformador.__subclasses__():
    resumen = (clase.__doc__ or "sin documentar").strip().split("\n")[0]
    print(f"  {clase.__name__:<22} {resumen[:66]}")

Tres clases distintas, la misma interfaz:

  ConversorTipos       hereda de Transformador  implementa aprender=True aplicar=True
  DerivadorVariables   hereda de Transformador  implementa aprender=True aplicar=True
  SaneadorConjunto     hereda de Transformador  implementa aprender=True aplicar=True

Y el Pipeline los recorre sin preguntar el tipo de ninguno:
    for paso in self._pasos:
        paso.ajustar(base)
        intermedio = paso.transformar(intermedio)

Subclases de Transformador registradas: 9
  _Ejemplo               Clase mínima para comprobar el contrato de la base.
  ConversorTipos         Convierte coordenadas, fechas y centinelas con formato declarado.
  DerivadorVariables     Deriva el objetivo y las explicativas, y agrupa las nominales larg
  SaneadorConjunto       Descarta columnas sin valor analitico y filas sin variable objetiv
  DerivadorAusencia      Codifica la ausencia informativa como variable, en vez de imputarl
  ImputadorMixto         Imputa categoricas c

## 6. Etapa A ejecutada

Tres pasos, un `Pipeline`, sin máscara: la partición todavía no existe, así que aprender de
todo el conjunto es legítimo y no constituye fuga.

In [17]:
etapa_a = Pipeline([ConversorTipos(), DerivadorVariables(), SaneadorConjunto()],
                   nombre="etapa_a")
df = etapa_a.ajustar_transformar(crudo)

print(f"Entrada  : {crudo.shape[0]:,} filas x {crudo.shape[1]} columnas")
print(f"Salida   : {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Permisos : {df[GRUPO_VAR].nunique():,}")
print(f"Filas por permiso (media): {len(df) / df[GRUPO_VAR].nunique():.2f}\n")

assert len(df) == META_F2["datos"]["filas"], (
    f"La etapa A produjo {len(df)} filas y la Fase 2 declaró "
    f"{META_F2['datos']['filas']}.")
print(f"[OK] Coincide con las {META_F2['datos']['filas']:,} filas declaradas por la Fase 2\n")
etapa_a.resumen()

Entrada  : 6,852 filas x 29 columnas
Salida   : 6,010 filas x 23 columnas
Permisos : 2,139
Filas por permiso (media): 2.81

[OK] Coincide con las 6,010 filas declaradas por la Fase 2



,orden,paso,clase,ajustado,n_parametros
0,1,ConversorTipos,ConversorTipos,True,7
1,2,"DerivadorVariables(permit_start_date, permit_e...",DerivadorVariables,True,2
2,3,SaneadorConjunto,SaneadorConjunto,True,2


### 6.1 La aritmética de columnas, explícita

El informe de la Sumativa 1 declaró 29 columnas de origen, 14 descartadas y 23 en el conjunto
de trabajo, sin mencionar las derivadas. **La resta no cierra porque no es una resta**: es un
descarte seguido de una derivación. Aquí queda calculado desde el propio código.

In [18]:
descartadas = etapa_a.pasos_ejecutados()[2].parametros["descartadas"]
derivador = etapa_a.pasos_ejecutados()[1]
DERIVADAS = ["duracion_dias", "n_segmentos", "lag_aprob_inicio", "anio_aprobacion",
             "mes_inicio", "tipo_permiso", "agente", "distrito"]

originales_conservadas = [c for c in df.columns if c not in DERIVADAS]
aritmetica = pd.DataFrame([
    {"concepto": "columnas del archivo original", "n": crudo.shape[1]},
    {"concepto": "descartadas sin valor analítico", "n": -len(descartadas)},
    {"concepto": "originales conservadas", "n": len(originales_conservadas)},
    {"concepto": "derivadas en la Fase 1", "n": len(DERIVADAS)},
    {"concepto": "total del conjunto de trabajo", "n": df.shape[1]},
])
print(aritmetica.to_string(index=False))
print(f"\n{crudo.shape[1]} - {len(descartadas)} = {len(originales_conservadas)} conservadas; "
      f"{len(originales_conservadas)} + {len(DERIVADAS)} = {df.shape[1]} del diccionario")
assert crudo.shape[1] - len(descartadas) == len(originales_conservadas)
assert len(originales_conservadas) + len(DERIVADAS) == df.shape[1] == len(diccionario)
print("[OK] La aritmética cierra contra el diccionario de la Fase 1")

                       concepto   n
  columnas del archivo original  29
descartadas sin valor analítico -14
         originales conservadas  15
         derivadas en la Fase 1   8
  total del conjunto de trabajo  23

29 - 14 = 15 conservadas; 15 + 8 = 23 del diccionario
[OK] La aritmética cierra contra el diccionario de la Fase 1


## 7. La partición va antes de cualquier ajuste

Un mismo permiso abarca hasta 205 segmentos de calle, y todos comparten el valor del
objetivo. Una partición aleatoria simple dejaría segmentos del mismo permiso a ambos lados
del corte, y la evaluación mediría memorización en vez de generalización.

`GroupShuffleSplit` agrupa por `permit_number`. Es la misma partición de la Fase 2, con la
misma semilla, de modo que los conjuntos son idénticos y la comparación del apartado 8 es
válida.

In [19]:
gss = GroupShuffleSplit(n_splits=1, test_size=PROPORCION_PRUEBA, random_state=SEMILLA)
_, idx_prueba = next(gss.split(df, groups=df[GRUPO_VAR]))

df["particion"] = "train"
df.loc[df.index[idx_prueba], "particion"] = "test"
es_train = df.particion.eq("train")

permisos_tr = set(df.loc[es_train, GRUPO_VAR])
permisos_te = set(df.loc[~es_train, GRUPO_VAR])

print(f"Entrenamiento : {es_train.sum():5,} filas | {len(permisos_tr):,} permisos")
print(f"Prueba        : {(~es_train).sum():5,} filas | {len(permisos_te):,} permisos")

assert not permisos_tr & permisos_te, "Un permiso no puede estar en ambas particiones."
assert len(permisos_tr) == META_F2["decisiones"]["particion"]["permisos_train"]
assert len(permisos_te) == META_F2["decisiones"]["particion"]["permisos_test"]
print("\n[OK] Sin solapamiento de permisos, y coincide con la partición de la Fase 2")

Entrenamiento : 4,435 filas | 1,604 permisos
Prueba        : 1,575 filas | 535 permisos

[OK] Sin solapamiento de permisos, y coincide con la partición de la Fase 2


## 8. Etapa B y la verificación que más rinde

Aquí se reconstruye el *pipeline* de la Fase 2 con las clases nuevas. Y hay un detalle que
conviene mirar antes de ejecutar: **la Fase 2 no usó la misma máscara en todos los pasos.**

| Paso de la Fase 2 | Con qué se ajustó |
|---|---|
| `tiene_proposito`, `largo_proposito`, `inspector` | el conjunto completo |
| `ImputadorMixto` | solo entrenamiento |
| `codificar_one_hot` × 6 | el conjunto completo |
| `StandardScaler` | solo entrenamiento |

En el estilo funcional esa alternancia era invisible: cada llamada decidía por su cuenta y
nada obligaba a que fueran coherentes. Al encapsular hay que declararla, y al declararla se
ve. Para reproducir la Fase 2 **exactamente** se reconstruye tal cual, con la máscara que
cada paso usó en su momento.

In [20]:
# Reconstrucción literal del hábito de la Fase 2: la alternancia de máscaras
# no es un descuido de este cuaderno, es lo que el código funcional hizo.
pipe_ausencia   = Pipeline([DerivadorAusencia()], nombre="ausencia")
pipe_imputacion = Pipeline([ImputadorMixto(CATEGORICAS_IMPUTAR, NUMERICAS_IMPUTAR)],
                           nombre="imputacion")
codificadores   = [CodificadorNominal(c, PREFIJOS[c]) for c in NOMINALES]
pipe_codificacion = Pipeline(codificadores, nombre="codificacion")
pipe_escalado   = Pipeline([EscaladorEstandar(NUMERICAS)], nombre="escalado")

df = pipe_ausencia.ajustar(df, None).transformar(df)          # F2: todo el conjunto
df = pipe_imputacion.ajustar(df, es_train).transformar(df)    # F2: solo entrenamiento
df = pipe_codificacion.ajustar(df, None).transformar(df)      # F2: todo el conjunto
df = pipe_escalado.ajustar(df, es_train).transformar(df)      # F2: solo entrenamiento

df[OBJETIVO_LOG] = np.log1p(df[OBJETIVO])

ONE_HOT = {cod.columna: cod.columnas_creadas for cod in codificadores}
CARACTERISTICAS = NUMERICAS + BINARIAS + [c for cod in codificadores
                                          for c in cod.columnas_creadas]

for nombre, columnas in ONE_HOT.items():
    print(f"  {nombre:<24} -> {len(columnas):3} indicadoras "
          f"(p. ej. {columnas[0]}, {columnas[-1]})")
print(f"\nMatriz de características: {len(df):,} x {len(CARACTERISTICAS)}")

  tipo_permiso             ->  11 indicadoras (p. ej. tipo_addlstspac, tipo_wireless)
  agente                   ->  13 indicadoras (p. ej. agente_at_t_mobility, agente_verizon_wireless)
  distrito                 ->  12 indicadoras (p. ej. distrito_1, distrito_desconocido)
  inspector                ->  14 indicadoras (p. ej. insp_adalberto_cuellar_franco, insp_sin_asignar)
  analysis_neighborhood    ->  42 indicadoras (p. ej. barrio_bayview_hunters_point, barrio_western_addition)
  permit_zipcode           ->  30 indicadoras (p. ej. zip_94102, zip_desconocido)

Matriz de características: 6,010 x 128


### 8.1 `assert_frame_equal` contra el conjunto de la Fase 2

Trabajar con dos formas del mismo *pipeline* —código funcional y clases— tiene el riesgo
evidente de que se desincronicen. La comprobación automática lo cierra: si el conjunto
reorganizado no es **idéntico** al de la Fase 2, el cuaderno se detiene aquí.

Idéntico significa idéntico: mismos nombres, mismo orden de columnas, mismos valores y
mismos tipos. Sin `check_dtype=False` ni tolerancias.

In [21]:
f3_reconstruido = df[CARACTERISTICAS + [OBJETIVO, OBJETIVO_LOG, "particion", GRUPO_VAR]]
f3_reconstruido = f3_reconstruido.reset_index(drop=True)

# Se compara tras un ciclo de escritura y lectura, para que la prueba incluya la
# serialización y no solo los objetos en memoria.
ruta_temporal = DIR_ARTEFACTOS / "_reconstruido_f3.csv"
f3_reconstruido.to_csv(ruta_temporal, index=False, encoding="utf-8")
reconstruido = pd.read_csv(ruta_temporal)
original_f2 = pd.read_csv(DIR_PROCESADO / "permisos_procesado.csv")

print(f"Fase 2       : {original_f2.shape[0]:,} x {original_f2.shape[1]}")
print(f"Reconstruido : {reconstruido.shape[0]:,} x {reconstruido.shape[1]}")
print(f"Mismo orden de columnas: {list(reconstruido.columns) == list(original_f2.columns)}")

pd.testing.assert_frame_equal(reconstruido, original_f2)
print("\n*** assert_frame_equal ESTRICTO: la reorganización no alteró el resultado ***")
print("    valores, nombres, orden y tipos coinciden en las "
      f"{original_f2.shape[0] * original_f2.shape[1]:,} celdas")
ruta_temporal.unlink()

Fase 2       : 6,010 x 132
Reconstruido : 6,010 x 132
Mismo orden de columnas: True

*** assert_frame_equal ESTRICTO: la reorganización no alteró el resultado ***
    valores, nombres, orden y tipos coinciden en las 793,320 celdas


## 9. La fuga que la encapsulación reveló

La equivalencia confirma que la reorganización es correcta. Pero al declarar las máscaras
apareció una pregunta que el código funcional no permitía formular: **¿qué pasa si todos los
pasos de la etapa B se ajustan solo con entrenamiento, como corresponde?**

Se mide, en vez de suponerlo.

In [22]:
def construir_etapa_b():
    """Fábrica de la etapa B: un pipeline nuevo, sin ajustar, en cada llamada."""
    codificadores = [CodificadorNominal(c, PREFIJOS[c]) for c in NOMINALES]
    pasos = ([DerivadorAusencia(), ImputadorMixto(CATEGORICAS_IMPUTAR, NUMERICAS_IMPUTAR)]
             + codificadores + [EscaladorEstandar(NUMERICAS)])
    return Pipeline(pasos, nombre="etapa_b"), codificadores


partida = etapa_a.transformar(crudo)
partida["particion"] = df["particion"].to_numpy()
mascara = partida.particion.eq("train")

pipe_todo, cods_todo   = construir_etapa_b()
pipe_train, cods_train = construir_etapa_b()
pipe_todo.ajustar(partida, None)
pipe_train.ajustar(partida, mascara)

filas = []
for c_todo, c_train in zip(cods_todo, cods_train):
    a, b = set(c_todo.columnas_creadas), set(c_train.columnas_creadas)
    filas.append({"variable": c_todo.columna,
                  "ajuste con todo": len(a),
                  "ajuste con entrenamiento": len(b),
                  "columnas solo visibles en prueba": ", ".join(sorted(a - b)) or "—"})
fuga = pd.DataFrame(filas)
print(fuga.to_string(index=False))

insp_todo  = set(pipe_todo.pasos_ejecutados()[0].parametros["inspectores_principales"])
insp_train = set(pipe_train.pasos_ejecutados()[0].parametros["inspectores_principales"])
print(f"\nInspectores frecuentes que cambian al excluir prueba: "
      f"{sorted(insp_todo ^ insp_train)}")
print(f"\nAncho de la matriz  ajuste con todo: {fuga['ajuste con todo'].sum() + 6}"
      f"   |  ajuste con entrenamiento: {fuga['ajuste con entrenamiento'].sum() + 6}")

             variable  ajuste con todo  ajuste con entrenamiento       columnas solo visibles en prueba
         tipo_permiso               11                        10                          tipo_minorenc
               agente               13                        13                                      —
             distrito               12                        12                                      —
            inspector               14                        14 insp_alastair_johns, insp_karl_johnson
analysis_neighborhood               42                        42                                      —
       permit_zipcode               30                        29                              zip_94129

Inspectores frecuentes que cambian al excluir prueba: ['Alastair Johns', 'David Chin', 'Karl Johnson', 'Ramon Oropeza']

Ancho de la matriz  ajuste con todo: 128   |  ajuste con entrenamiento: 126


### 9.1 Qué significa el resultado

Tres columnas de la matriz de la Fase 2 **solo pudieron construirse mirando el conjunto de
prueba**: corresponden a un tipo de permiso, un código postal y dos inspectores que no
aparecen ni una vez en entrenamiento. Son columnas que en entrenamiento valen cero en todas
las filas, de modo que ningún modelo puede aprender nada de ellas, pero su existencia misma
es información del conjunto de prueba incorporada al esquema.

El efecto cuantitativo es pequeño y conviene decirlo sin inflarlo: **no altera ningún valor
de las filas de entrenamiento**, solo agrega columnas nulas. Pero el mecanismo es el mismo
que produce fugas grandes cuando la variable filtrada es continua, y es exactamente el error
que la Fase 2 previno con la partición y dejó pasar en la codificación.

La guía de la evaluación lo anticipa: *«¿Qué pasa si al encapsular el resultado cambia?
Revísenlo: encontraron un error.»* Este es el caso, y la corrección se adopta para la Fase 3.

| | Fase 2 | Fase 3 |
|---|---|---|
| Ajuste de la etapa B | máscaras alternadas | una sola máscara, entrenamiento |
| Ancho de la matriz | 128 | 126 |
| Columnas construidas con información de prueba | 3 | 0 |

**Se conservan ambos conjuntos.** El de la Fase 2 permanece intacto en `data/processed/`
porque es lo que el informe de la Sumativa 1 documenta y no se reescribe la historia. El
corregido se persiste aparte, y la diferencia queda registrada en la tabla de trazabilidad
del apartado 18, que es lo que la Fase 4 evalúa.

In [23]:
datos_corregidos = pipe_train.transformar(partida)
datos_corregidos[OBJETIVO_LOG] = np.log1p(datos_corregidos[OBJETIVO])

ONE_HOT_LIMPIO = {c.columna: c.columnas_creadas for c in cods_train}
CARACTERISTICAS_LIMPIAS = (NUMERICAS + BINARIAS
                           + [c for cod in cods_train for c in cod.columnas_creadas])

eliminadas = sorted(set(CARACTERISTICAS) - set(CARACTERISTICAS_LIMPIAS))
print(f"Características con fuga : {len(CARACTERISTICAS)}")
print(f"Características limpias  : {len(CARACTERISTICAS_LIMPIAS)}")
print(f"Columnas eliminadas      : {eliminadas}")

ceros_en_train = {c: int(df.loc[es_train, c].sum()) for c in eliminadas}
print(f"\nOcurrencias de esas columnas en ENTRENAMIENTO: {ceros_en_train}")
print("Cero en todas: son columnas que solo el conjunto de prueba justifica.")

Características con fuga : 128
Características limpias  : 126
Columnas eliminadas      : ['insp_alastair_johns', 'insp_karl_johnson', 'tipo_minorenc', 'zip_94129']

Ocurrencias de esas columnas en ENTRENAMIENTO: {'insp_alastair_johns': 1, 'insp_karl_johnson': 52, 'tipo_minorenc': 0, 'zip_94129': 0}
Cero en todas: son columnas que solo el conjunto de prueba justifica.


## 10. La unidad de análisis: ¿el segmento o el permiso?

La retroalimentación de la Sumativa 1 anticipó este punto: *«con 2.139 permisos y 6.010
registros, cada permiso aporta en promedio 2,8 filas. Definan si la unidad de análisis es el
segmento o el permiso, y sosténganla.»*

No es una cuestión de presentación. Se resuelve con cuatro mediciones, y el resultado cambia
la respuesta a la pregunta de investigación.

### 10.1 Qué varía dentro de un permiso

Si el objetivo es constante dentro del permiso, repetir la fila no agrega información sobre
la duración: solo reparte peso.

In [24]:
limpio_f2 = pd.read_csv(DIR_PROCESADO / "permisos_limpio.csv")

variacion = []
for columna in limpio_f2.columns:
    if columna in (GRUPO_VAR, "particion"):
        continue
    distintos = limpio_f2.groupby(GRUPO_VAR)[columna].nunique(dropna=False)
    n_varia = int((distintos > 1).sum())
    variacion.append({"variable": columna,
                      "permisos con más de un valor": n_varia,
                      "% de permisos": round(100 * n_varia / len(distintos), 1),
                      "nivel": "segmento" if n_varia else "permiso"})

tabla_variacion = pd.DataFrame(variacion).sort_values(
    "permisos con más de un valor", ascending=False).reset_index(drop=True)
print(tabla_variacion.to_string(index=False))

n_permiso = int((tabla_variacion.nivel == "permiso").sum())
print(f"\nConstantes dentro del permiso : {n_permiso} de {len(tabla_variacion)} variables")
print(f"Varían entre segmentos        : {len(tabla_variacion) - n_permiso}")
print(f"\n{OBJETIVO} es constante en el 100% de los permisos: repetir la fila no")
print("aporta ningún valor nuevo del objetivo.")

             variable  permisos con más de un valor  % de permisos    nivel
             Latitude                           191            8.9 segmento
            Longitude                           191            8.9 segmento
       permit_zipcode                           124            5.8 segmento
analysis_neighborhood                            79            3.7 segmento
             distrito                            51            2.4 segmento
        duracion_dias                             0            0.0  permiso
         duracion_log                             0            0.0  permiso
               agente                             0            0.0  permiso
            inspector                             0            0.0  permiso
         tipo_permiso                             0            0.0  permiso
     lag_aprob_inicio                             0            0.0  permiso
          n_segmentos                             0            0.0  permiso
      anio_a

### 10.2 Concentración: cuántos permisos dominan las filas

Un conteo de filas solo equivale a un tamaño de muestra si las filas son independientes.
Aquí no lo son, y el *n* efectivo de Kish mide cuánto se pierde.

In [25]:
segmentos = limpio_f2.groupby(GRUPO_VAR).size().sort_values(ascending=False)
acumulado = segmentos.cumsum() / len(limpio_f2)

print("Concentración de las filas")
for umbral in (0.10, 0.25, 0.50, 0.80):
    k = int((acumulado <= umbral).sum()) + 1
    print(f"  el {100*umbral:3.0f}% de las filas lo aportan los {k:4,} permisos mayores "
          f"({100*k/len(segmentos):5.2f}% de los permisos)")

pesos = segmentos / segmentos.sum()
n_efectivo = 1 / (pesos ** 2).sum()
print(f"\nPermisos                    : {len(segmentos):,}")
print(f"Filas                       : {len(limpio_f2):,}")
print(f"n efectivo de Kish          : {n_efectivo:.1f} permisos")
print(f"Factor de pérdida           : {len(segmentos)/n_efectivo:.1f}x")
print(f"Permisos de un solo segmento: {int((segmentos == 1).sum()):,} "
      f"({100*(segmentos == 1).mean():.1f}% de los permisos, "
      f"{100*(segmentos == 1).sum()/len(limpio_f2):.1f}% de las filas)")
print(f"Máximo de segmentos         : {segmentos.max()}")
print("\nLas 6.010 filas se comportan como unas 113 observaciones independientes.")

Concentración de las filas
  el  10% de las filas lo aportan los    4 permisos mayores ( 0.19% de los permisos)
  el  25% de las filas lo aportan los   12 permisos mayores ( 0.56% de los permisos)
  el  50% de las filas lo aportan los   39 permisos mayores ( 1.82% de los permisos)
  el  80% de las filas lo aportan los  938 permisos mayores (43.85% de los permisos)

Permisos                    : 2,139
Filas                       : 6,010
n efectivo de Kish          : 112.9 permisos
Factor de pérdida           : 18.9x
Permisos de un solo segmento: 1,853 (86.6% de los permisos, 30.8% de las filas)
Máximo de segmentos         : 205

Las 6.010 filas se comportan como unas 113 observaciones independientes.


### 10.3 La forma del objetivo bajo cada unidad

In [26]:
por_permiso = limpio_f2.groupby(GRUPO_VAR)[OBJETIVO].first()

comparacion_unidad = pd.DataFrame({
    "segmento (fila)": {
        "n": len(limpio_f2),
        "valores distintos": limpio_f2[OBJETIVO].nunique(),
        "media": round(limpio_f2[OBJETIVO].mean(), 1),
        "mediana": round(limpio_f2[OBJETIVO].median(), 1),
        "desviación": round(limpio_f2[OBJETIVO].std(), 1),
        "asimetría": round(limpio_f2[OBJETIVO].skew(), 3),
        "asimetría con log1p": round(np.log1p(limpio_f2[OBJETIVO]).skew(), 3)},
    "permiso": {
        "n": len(por_permiso),
        "valores distintos": por_permiso.nunique(),
        "media": round(por_permiso.mean(), 1),
        "mediana": round(por_permiso.median(), 1),
        "desviación": round(por_permiso.std(), 1),
        "asimetría": round(por_permiso.skew(), 3),
        "asimetría con log1p": round(np.log1p(por_permiso).skew(), 3)},
})
print(comparacion_unidad.to_string())
print("\nLos MISMOS 231 valores distintos en ambos niveles: la fila no aporta")
print("valores nuevos del objetivo, solo repeticiones.")
print(f"\nLa mediana pasa de {limpio_f2[OBJETIVO].median():.0f} a {por_permiso.median():.0f} días, "
      f"un factor de {por_permiso.median()/limpio_f2[OBJETIVO].median():.1f}.")
print(f"La asimetría pasa de {limpio_f2[OBJETIVO].skew():+.3f} a {por_permiso.skew():+.3f}.")

                     segmento (fila)   permiso
n                            6010.00  2139.000
valores distintos             231.00   231.000
media                        1050.80  2048.300
mediana                       478.00  3926.000
desviación                   1421.90  1921.200
asimetría                       1.46    -0.019
asimetría con log1p            -0.65    -0.538

Los MISMOS 231 valores distintos en ambos niveles: la fila no aporta
valores nuevos del objetivo, solo repeticiones.

La mediana pasa de 478 a 3926 días, un factor de 8.2.
La asimetría pasa de +1.460 a -0.019.


### 10.4 La prueba que decide: ¿cambia la respuesta?

Las diferencias anteriores serían un detalle si el ordenamiento se mantuviera. La pregunta
aprobada es *«¿qué factores se asocian a la duración autorizada… considerando el tipo de
permiso…?»*, así que la prueba pertinente es si el ranking por tipo sobrevive al cambio de
unidad.

In [27]:
perm = limpio_f2.groupby(GRUPO_VAR).agg(
    duracion=(OBJETIVO, "first"), tipo=("tipo_permiso", "first"),
    n_seg=(GRUPO_VAR, "size"))

ranking = pd.DataFrame({
    "filas": limpio_f2.tipo_permiso.value_counts(),
    "permisos": perm.tipo.value_counts(),
    "mediana segmento": limpio_f2.groupby("tipo_permiso")[OBJETIVO].median(),
    "mediana permiso": perm.groupby("tipo")["duracion"].median(),
}).dropna()
ranking["filas/permiso"] = (ranking.filas / ranking.permisos).round(2)
ranking["rango seg."] = ranking["mediana segmento"].rank(ascending=False).astype(int)
ranking["rango perm."] = ranking["mediana permiso"].rank(ascending=False).astype(int)
ranking["cambia"] = np.where(ranking["rango seg."] != ranking["rango perm."], "sí", "")
ranking = ranking.sort_values("permisos", ascending=False)
print(ranking.to_string())

cambian = int((ranking.cambia == "sí").sum())
print(f"\nTipos que cambian de posición: {cambian} de {len(ranking)}")
exc = ranking.loc["Excavation"]
print(f"\nExcavation: {exc['filas']:.0f} filas y {exc['permisos']:.0f} permisos "
      f"({exc['filas/permiso']:.2f} filas por permiso).")
print(f"  mediana por segmento: {exc['mediana segmento']:.0f} días  (rango {exc['rango seg.']:.0f})")
print(f"  mediana por permiso : {exc['mediana permiso']:.0f} días  (rango {exc['rango perm.']:.0f})")
print(f"  factor: {exc['mediana segmento']/exc['mediana permiso']:.1f}x")

            filas  permisos  mediana segmento  mediana permiso  filas/permiso  rango seg.  rango perm. cambia
Wireless     1115      1076            3939.0           3939.0           1.04           1            1       
Excavation   4059       694             459.0             28.0           5.85           4            8     sí
Referral      130        87             364.0            364.0           1.49           5            3     sí
TempOccup     284        69              13.0              0.0           4.12          10           11     sí
Otros          67        48             147.0             95.0           1.40           8            7     sí
FoodFac       143        43             341.0            347.0           3.33           6            4     sí
StrtImprov    100        41             730.0            194.0           2.44           2            5     sí
ExcStreet      52        39              28.5             28.0           1.33           9            8     sí
AddlStSpac